# Selenium

In [73]:
from bs4 import BeautifulSoup
import requests
import re
import pandas as pd
import csv
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import html5lib

In [74]:
#PATH = "G:\\User\\Documents\\Chuyen de\\Edge Driver\\msedgedriver.exe"
eService = webdriver.EdgeService(executable_path="G:\\User\\Documents\\Chuyen de\\Edge Driver\\msedgedriver.exe")

In [75]:
driver = webdriver.Edge(service=eService)

driver.get("https://www.imdb.com/search/title/?release_date=1900-01-01,2025-01-01")

print(driver.title)
#button = driver.find_element(By.CLASS_NAME,"ipc-see-more__text")

n_loop = 0 #n_data = 50 + n_loop*50
csv_filename = 'MovieOverview.csv'

for i in range(n_loop):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight-1000)")
    time.sleep(5)
    button = WebDriverWait(driver,10).until(
        EC.presence_of_element_located((By.CLASS_NAME,"ipc-see-more__text"))
        )
    button.click()
    time.sleep(10)

Release date between 1900-01-01 and 2025-01-01 (Sorted by Popularity Ascending)


In [76]:
url = driver.page_source
soup = BeautifulSoup(url, "html.parser")
movies =  soup.find_all('li', class_='ipc-metadata-list-summary-item')
print(len(movies))

50


In [77]:
#for movie in movies:
#    link='https://www.imdb.com/'+movie.find('a', class_='ipc-title-link-wrapper',href=True).attrs['href'].split('?')[0]
#    print(link)

In [78]:
csv_filename = 'MovieOverview.csv'

In [79]:
def get_directors(url):
    credit_url = url
    df=pd.read_html(credit_url)[0]
    df=df[0].head(5)
    df=str(df.tolist())[1:-1]
    df=df.replace("'","")
    return df

In [80]:
def get_writer(url):
    credit_url = url
    df=pd.read_html(credit_url)[1]
    df=df[0].head(5)
    df=str(df.tolist())[1:-1]
    df=df.replace("'","")
    return df

In [90]:
def get_cast(url):
    credit_url = url
    df= pd.read_html(credit_url)[2]
    df=df[1].dropna().head(3)
    df=str(df.tolist())[1:-1]
    df=df.replace("'","")
    return df

In [145]:
def get_genre(url):
    driver = webdriver.Edge(service=eService)
    movie_url = url
    driver.get(movie_url)
    movie_url = driver.page_source
    soup = BeautifulSoup(movie_url, "html.parser")
    genres =  soup.find_all('a',class_='ipc-chip ipc-chip--on-baseAlt')
    driver.close()
    lists = []
    for genre in genres:
        lists.append(genre.text)
    df=str(lists)[1:-1]
    df=df.replace("'","")
    return df

In [146]:
with open(csv_filename, mode='w', newline='',encoding='utf-8-sig') as file:
    writer=csv.writer(file)
    header = ['Name', 'Year', 'Rating','Overview','Movie Link','Review Link', 'Credit Link', 'Director(s)','Writer(s)','Cast','Genres']
    writer.writerow(header)
    for movie in movies:
        name=movie.find('div',class_='ipc-title ipc-title--base ipc-title--title ipc-title-link-no-icon ipc-title--on-textPrimary sc-a69a4297-2 bqNXEn dli-title with-margin').a.text.split('.')[1]
        year=movie.find('div',class_='sc-300a8231-6 dBUjvq dli-title-metadata').span.text
        rating=movie.find('div',class_='sc-e2dbc1a3-0 jeHPdh sc-300a8231-3 koIPa dli-ratings-container').span.text.split('(')[0]
        overview=movie.find('div',class_='ipc-html-content ipc-html-content--base sc-2bfd043a-0 bktZnv dli-plot-container').text
        movie_link='https://www.imdb.com/'+movie.find('a', class_='ipc-title-link-wrapper',href=True).attrs['href'].split('?')[0]
        movie_review_link='https://www.imdb.com/'+movie.find('a', class_='ipc-title-link-wrapper',href=True).attrs['href'].split('?')[0]+'reviews'
        movie_credit_link='https://www.imdb.com/'+movie.find('a', class_='ipc-title-link-wrapper',href=True).attrs['href'].split('?')[0]+'fullcredits'
        directors = get_directors(movie_credit_link)
        writers = get_writer(movie_credit_link)
        cast = get_cast(movie_credit_link)
        genre = get_genre(movie_link)
        writer.writerow([name, year, rating, overview, movie_link, movie_review_link, movie_credit_link, directors, writers, cast, genre])

# Soup Maxxing

In [89]:
credit_url = 'https://www.imdb.com//title/tt0898266/fullcredits'
df= pd.read_html(credit_url)[2]
df=df[1].dropna()
df=str(df.tolist())[1:-1]
df=df.replace("'","")
print(df)

Johnny Galecki, Jim Parsons, Kaley Cuoco, Simon Helberg, Kunal Nayyar, Melissa Rauch, Mayim Bialik, Kevin Sussman, Carol Ann Susi, John Ross Bowie, Laura Spencer, Wil Wheaton, Brian George, Christine Baranski, Brian Posehn, Laurie Metcalf, Joshua Malina, Ian Scott Rudolph, Aarti Mann, Brian Thomas Smith, Alice Amter, Pamela Adlon, Sara Gilbert, Rati Gupta, Kate Micucci, Lauren Lapkus, Stephen Hawking, Regina King, Bob Newhart, Casey Sander, Dean Norris, Michael Massimino, Vernee Watson, Mark Harelik, Keith Carradine, Alessandra Torresani, Pasha D. Lychnikoff, Ed Lieberman, Robert Clotworthy, Margo Harshman, Sara Rue, Brian Patrick Wade, LeVar Burton, Stephen Merchant, Riki Lindhome, Kathy Bates, Kimberly Brooks, Ira Flatow, Swati Kapila, Marcus Folmar, "Jerry OConnell", Kal Penn, Teller, Sean Astin, James Hong, "Erin Allin OReilly", Lio Tipton, Molly Morgan, Neil deGrasse Tyson, Katee Sackhoff, Elena Campbell-Martinez, David Trice, Kevin Brief, Lance Barber, Dakin Matthews, Stephen Roo

In [59]:
credit_url = 'https://www.imdb.com//title/tt0898266/fullcredits'
df=pd.read_html(credit_url)[1]
df=df[0].head(5)
df=str(df.tolist())[1:-1]
df=df.replace("'","")
print(df)

Chuck Lorre, Chuck Lorre, Chuck Lorre, Chuck Lorre, Bill Prady


In [60]:
credit_url = 'https://www.imdb.com//title/tt0898266/fullcredits'
df=pd.read_html(credit_url)[0]
df=df[0].head(5)
df=str(df.tolist())[1:-1]
df=df.replace("'","")
print(df)

Mark Cendrowski, Anthony Rich, Peter Chakos, Nikki Lorre, James Burrows


In [142]:
driver = webdriver.Edge(service=eService)
movie_url = 'https://www.imdb.com/title/tt0898266'
driver.get(movie_url)
movie_url = driver.page_source
soup = BeautifulSoup(movie_url, "html.parser")
genres =  soup.find_all('a',class_='ipc-chip ipc-chip--on-baseAlt')
lists = []
for genre in genres:
    lists.append(genre.text)
df=str(lists)[1:-1]
df=df.replace("'","")
print(df)


Sitcom, Comedy, Romance
